#### Lab Exercise 2

**max marks = 30**

1. In this exercise and the next you will build a NB classifier for *sentiment analysis*.
2. The purpose of this part is to calculate probabilities of tokens in different class.
3. The dataset is a collection of sms texts, in a **csv** file.
4. We will use word tokens, consisting of lowercase English letters only, for simplicity.
5. The following cells contain templates of the functions to be completed.

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
#gutenberg_files = gutenberg.fileids()

Load Data

In [3]:
df = pd.read_csv("Emotion_classify_Data.csv")

In [4]:
len(df)

5937

In [5]:
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [6]:
df['Emotion'].value_counts()

,count
Emotion,
anger,2000
joy,2000
fear,1937


**Task 1.** *Complete the function "prepare_data".*  **[10 marks]**
1. The function takes a file input and float argument *x* in the range $0.6 \leq x \leq 0.9$ and the output are two dataframes.
2. First, process the full dataframe and replace the "negative" emotions (*anger*, *fear*) by 0 and positive emotion (*joy*) by 1.
3. The two dataframes are for training and testing the model. The fraction *x* is the fraction of 'documents' for training. For example, if $x = .8$ then (about) 80% of the texts are used for the training and the rest for training. The selection should be random.
4. You must complete function using basic Python libraries (*pandas*, *numpy* etc.) and should **not** use platforms like scikit-learn.

In [7]:
fl = "Emotion_classify_Data.csv"

In [8]:
def prepare_data(fl:str,  x:float) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
      The function prepares the data for training and testing.

      Args:
        fl (str): The file path to the CSV dataset.
        x (float): The fraction of data to be used for training, in the range [0.6, 0.9].

      Returns:
        tuple: A tuple containing two pandas DataFrames:
               - df_tr (DataFrame): The training dataset.
               - df_ts (DataFrame): The testing dataset.
    """
    # Read the dataset
    df = pd.read_csv(fl).copy()

    # Convert the labels
    df['Emotion'] = df['Emotion'].apply(lambda x: 0 if x in ['anger', 'fear'] else 1)

    # Clean text
    def clean_text(text):
        text = text.lower() # converts text into lower case.
        text = re.sub(r'[^a-z\s]', '', text) # removes punctuation
        return text

    df['Comment'] = df['Comment'].apply(clean_text)

    # Shuffle dataset before splitting
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # Train and test split
    train_size = int(len(df) * x)

    df_tr = df.iloc[:train_size] #dataframe for training data
    df_ts = df.iloc[train_size:] #dataframe for test data

    return df_tr, df_ts

In [9]:
df.columns

Index(['Comment', 'Emotion'], dtype='object')

**Task 2.** *Complete the function "build_vocb_counts" below.*  **[15 marks]**
1. The function takes a *dataframe* as input. The ouput is dictionary whose keys are tuples consisting of a token and the label (0 or 1). For example, if the word 'bad' occurs in a document with positive emotion, then the key i ('bad', 1) and if it occurs in document of negative emotion then the key is ('bad', 0). The values in the dictionary are the counts of token.
2. The count of token is the number of occurrences in all documents. Note that for the same token, there will be two counts (one for positive-emotion and one for negative-emotion). Use add-one smoothing. This means that the default count is 1. So, if the actual count of  ('bad', 0), is 10 the dictionary entry is 10+1 = 11.
3. Use *defaultdict*().
4. Use only the dataframe for training.

In [10]:
#The output of this function is a dictionary.
from collections import defaultdict
def build_vocb_counts(df: pd.DataFrame) -> defaultdict:
    """
      This function reads every comment, breaks it into words, and counts how
      often each word appears for each emotion label.

      Args:
        df (pd.DataFrame): The input DataFrame containing comments and emotion labels.

      Returns:
          d (defaultdict): A defaultdict containing word-emotion counts.
              (token, label) -> count
    """
    d = defaultdict(lambda: 1) # add-one smoothing

    for _, row in df.iterrows():
      text = row['Comment']
      label = row['Emotion']

      tokens = re.findall(r'[a-z]+', text.lower())

      for token in tokens:
        d[(token, label)] += 1

    return d

**Task 3.** *Complete the function "calculate_prob" below*. **[5 marks]**
1. The function takes a dictionary as input and outputs another dictionary.
2. The output dictionary contains the conditional probabilities *P(token|label)*. Given a token, what is its probability in class 0 and class 1, respectively? For example, *P('bad'|0)*.
3. You need to calculate for one class only (why?).
4. You will use these functions to build a naive Bayes classifier for emotions.
5. Use the vocabulary dictionary.

correct:
def calculate_prob(vocab_counts: defaultdict):

In [11]:
def calculate_prob(d: defaultdict) -> dict:
    """
      Function to calculate conditional probabilities P(token | label)
      from a vocabulary-count dictionary.

      Args:
        d : A dictionary where
          (token, label) -> count

      Returns:
          tuple:
              probs (dict): (token, label) -> probability
              total_count (dict): total token counts for each label
    """
    probs = {}

    total_count = {0: 0, 1: 0}

    for (_, label), count in d.items():
        total_count[label] += count

    for (token, label), count in d.items():
        probs[(token, label)] = count / total_count[label]

    return probs, total_count

In [12]:
#df_tr, df_ts = prepare_data(fl, 0.8)
# vocab_count = build_vocb_counts(df_tr)
# calculate_prob(vocab_count)

Build Classifier

In [13]:
def predict_comment(text, probs, total_count, class_priors):
    tokens = re.findall(r"[a-z]+", text.lower())

    score_0 = np.log(class_priors[0])
    score_1 = np.log(class_priors[1])

    for token in tokens:
      prob_0 = probs.get((token, 0), 1 / total_count[0])
      prob_1 = probs.get((token, 1), 1 / total_count[1])

      score_0 += np.log(prob_0)
      score_1 += np.log(prob_1)

    return 1 if score_1 > score_0 else 0


def predict_test_set(df_test, probs, total_count, class_priors):
    predictions = []

    for _, row in df_test.iterrows():
        predictions.append(
            predict_comment(row["Comment"], probs, total_count, class_priors)
        )

    return predictions


def accuracy_score(y_true, y_pred):
    correct = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
    return correct / len(y_true)

Run the full pipeline

In [14]:
df_train, df_test = prepare_data(fl, 0.8)

vocab_counts = build_vocb_counts(df_train)
probs, total_count = calculate_prob(vocab_counts)

class_priors ={
    0: (df_train["Emotion"] == 0).mean(),
    1: (df_train["Emotion"] == 1).mean()
}

y_pred = predict_test_set(df_test, probs, total_count, class_priors)
y_true = df_test["Emotion"].tolist()

accuracy = accuracy_score(y_true, y_pred)

In [15]:
print(accuracy)

0.9132996632996633


In [16]:
class_priors

{0: np.float64(0.6647715308485997), 1: np.float64(0.3352284691514003)}